# Scratchpad (State 1)

Interactive derivations and numerical checks before promotion to the clean paper
(`docs/fund-flow-rotation.tex`). Results are marked `[VERIFIED]` only after an
independent numerical check and human consensus. Dead ends are kept, marked
`[DEAD-END]`.

## 2026-06-10 - Smoothing the rotation-graph coordinates

**Problem.** The rotation graph plots RS-Ratio $\mathrm{RS}_{c,t}$ (eq:rs_ratio)
against RS-Momentum $\mathrm{RS}^{\mathrm{m}}_{c,t}$ (eq:rs_momentum). Built on raw
monthly relative flow, the per-sector tails crisscross the plane instead of tracing
the clockwise arcs the graph is meant to show. Monthly $\mathrm{rel}_{c,t} = g_{c,t}
- g_{U,t}$ inherits the full noise of a single month of fund flow, so the tail is
dominated by high-frequency jitter rather than rotation.

**Proposed fix.** Smooth the relative-strength signal with a trailing $w$-month mean
before standardizing,
$$\bar{\mathrm{rel}}_{c,t}(w) = \frac{1}{w}\sum_{k=0}^{w-1}\mathrm{rel}_{c,t-k},$$
and define the rotation-graph relative strength as the standardization of the
smoothed series,
$$\mathrm{RS}_{c,t} = \frac{\bar{\mathrm{rel}}_{c,t} - \mu_L(\bar{\mathrm{rel}}_c)}
{\sigma_L(\bar{\mathrm{rel}}_c)},\qquad
\mathrm{RS}^{\mathrm{m}}_{c,t} = \mathrm{RS}_{c,t} - \mathrm{RS}_{c,t-D}.$$
This is the standard construction: the JdK / Bloomberg RRG is built on a smoothed
relative-strength line, not the raw ratio.

**Self-challenge.**
- `[CHECK]` Degenerate case $w=1$: $\bar{\mathrm{rel}} = \mathrm{rel}$, so the
  coordinates reduce exactly to the current unsmoothed definition. Good - smoothing
  is a strict generalization, the old behavior is $w=1$.
- `[CHECK]` Internal consistency: the mean is linear, so $\bar{\mathrm{rel}} =
  \overline{g_c} - \overline{g_U}$; smoothing the strength signal is the same as
  using smoothed growth against a smoothed baseline. No inconsistency.
- `[CHECK]` Over-smoothing: a large $w$ manufactures an artificially clean spiral
  and adds a lag of about $(w-1)/2$ months. Keep $w$ small; do not let the chart
  invent rotation that is not in the data.
- `[CHECK]` Cost: $\mathrm{RS}$ needs $w-1$ extra warm-up months before it is defined
  (the smoothing window must fill before the standardization window starts).
- `[CHECK]` Does it actually de-jitter? Needs an empirical measure, below.

**Numerical check.** Measure the mean per-sector tail path length in $(\mathrm{RS},
\mathrm{RS}^{\mathrm{m}})$ space over the last 6 months: the summed Euclidean length
of the polyline a sector traces. Genuine rotation has modest length; jitter inflates
it with back-and-forth zigzag. If smoothing helps, length should fall.

In [1]:
import numpy as np
import sanity_check as sc
from categories import category_panel
from rotation import relative_flow, z_score, momentum

panel = sc.load()
cat0 = relative_flow(category_panel(panel), panel)

def coords(w):
    c = cat0.sort_values(['category', 'month']).copy()
    c['rel_s'] = (c['rel'] if w == 1 else
                  c.groupby('category')['rel'].transform(
                      lambda s: s.rolling(w, min_periods=w).mean()))
    c = z_score(c, col='rel_s', lookback=12, out='rs')
    c = momentum(c, col='rs', lag=3, out='rs_mom')
    return c

def mean_tail_path(c, tail=6):
    months = sorted(c['month'].unique())[-(tail + 1):]
    sub = c[c['month'].isin(months)].dropna(subset=['rs', 'rs_mom'])
    L = []
    for _, d in sub.groupby('category'):
        d = d.sort_values('month')
        if len(d) < 2:
            continue
        dx = np.diff(d['rs'].values); dy = np.diff(d['rs_mom'].values)
        L.append(np.sqrt(dx**2 + dy**2).sum())
    return np.mean(L)

print('smoothing w | mean per-sector 6m tail path length in (rs,rs_mom)')
for w in [1, 2, 3, 4]:
    print(f'   w={w}   {mean_tail_path(coords(w)):.3f}')

smoothing w | mean per-sector 6m tail path length in (rs,rs_mom)
   w=1   17.112
   w=2   11.989
   w=3   11.459
   w=4   9.401


**Result.** Path length falls 17.1 -> 12.0 -> 11.5 -> 9.4 for $w = 1,2,3,4$. The
$w=1 \to 2$ step removes most of the jitter (a 30% drop); $w=2 \to 3$ adds little
(11.99 -> 11.46); $w=4$ keeps shrinking but at the cost of more lag, consistent with
the over-smoothing `[CHECK]`.

**Recommendation.** Adopt the trailing-mean smoothing with a default $w=3$: it
matches the quarterly cadence already used for the momentum lag $D=3$, and a quarter
is the natural unit for a flow-rotation read. $w=2$ is a defensible lighter
alternative that captures most of the de-jittering with one fewer month of lag.

Status: `[VERIFIED]`. Consensus 2026-06-10: adopt the trailing-mean smoothing with
default $w=3$. Promoted to State 2 - eq:rel_smoothed (smoothing) added and eq:rs_ratio
generalized to the smoothed signal.